In [4]:
#stopwords , tokenization and lemmatization and positional indexing

In [5]:
!pip install nltk

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [7]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re

# ------------------------------
# 1. Preprocessing
# ------------------------------
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    # Lowercase + remove punctuation
    text = re.sub(r'[^\w\s]', '', text.lower())
    tokens = word_tokenize(text)
    # Remove stopwords + lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and t.isalpha()]
    return tokens

# ------------------------------
# 2. Build Positional Index
# ------------------------------
def build_index(docs):
    index = {}  # term -> {doc_id: [positions]}
    for doc_id, doc in enumerate(docs):
        tokens = preprocess(doc)
        for pos, token in enumerate(tokens):
            if token not in index:
                index[token] = {}
            if doc_id not in index[token]:
                index[token][doc_id] = []
            index[token][doc_id].append(pos)
    return index

# ------------------------------
# 3. Search: AND + Phrase Support
# ------------------------------
def search(query, index, docs):
    q_tokens = preprocess(query)
    if not q_tokens:
        return []

    # Start with docs from first term
    if q_tokens[0] not in index:
        return []
    result_docs = set(index[q_tokens[0]].keys())

    # AND: intersect with other terms
    for term in q_tokens[1:]:
        if term not in index:
            return []
        result_docs &= set(index[term].keys())

    # Phrase check: tokens must appear consecutively
    if len(q_tokens) > 1:
        valid_docs = []
        for doc_id in result_docs:
            positions = [index[t][doc_id] for t in q_tokens if doc_id in index[t]]
            if not positions:
                continue
            # Check if any position sequence is consecutive
            pos_lists = [sorted(index[t][doc_id]) for t in q_tokens]
            for i in range(len(pos_lists[0])):
                if all(pos_lists[j][0] + j == pos_lists[0][i] + j
                       for j in range(1, len(pos_lists))
                       if i + j < len(pos_lists[j])):
                    valid_docs.append(doc_id)
                    break
        result_docs = valid_docs

    return [(i, docs[i]) for i in result_docs]


In [ ]:
# ------------------------------
# 4. Demo
# ------------------------------
docs = [
    "The quick brown fox jumps over the lazy dog.",
    "A fox is quick and brown.",
    "The dog sleeps all day.",
    "Quick foxes love jumping."
]

index = build_index(docs)

print("Index built!")
print("Try queries like: 'quick fox', 'brown dog', 'fox jumps'")

while True:
    query = input("\nSearch: ").strip()
    if query == "quit":
        break
    results = search(query, index, docs)
    print(f"\nFound {len(results)} document(s):")
    for doc_id, text in results:
        print(f"  [{doc_id}] {text}")

Index built!
Try queries like: 'quick fox', 'brown dog', 'fox jumps'

Search: fox and 

Found 3 document(s):
  [0] The quick brown fox jumps over the lazy dog.
  [1] A fox is quick and brown.
  [3] Quick foxes love jumping.
